# 📊 Analyse de Données de Vente – Online Retail Dataset
**Big Data Project – PySpark + HDFS**

## 1️⃣ Initialisation de la session Spark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Analyse Ventes Online Retail") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Spark démarré avec succès !")
print(f"Version Spark : {spark.version}")

✅ Spark démarré avec succès !
Version Spark : 3.5.0


## 2️⃣ Chargement des données
> Le fichier `online_retail.csv` (≈ 542 000 lignes) est monté dans le conteneur sous `/home/jovyan/data/`.
> Il est généré à partir de `Online Retail.xlsx` (voir `scripts/convert_to_csv.py`).

In [2]:
# Chargement depuis le fichier local (dossier data/)
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", "\"") \
    .csv("/home/jovyan/data/online_retail.csv")

print(f"✅ Données chargées : {df.count()} lignes")
print(f"📋 Colonnes : {df.columns}")
df.printSchema()

✅ Données chargées : 541909 lignes
📋 Colonnes : ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [3]:
# Aperçu des premières lignes
df.show(5, truncate=False)

+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |2010-12-01 08:26:00|2.55     |17850.0   |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |2010-12-01 08:26:00|2.75     |17850.0   |United Kingdom|
|536365   |84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
|536365   |84029E   |RED WOOLLY HOTTIE WHITE HEART.     |6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
+---------+-----

## 3️⃣ Nettoyage des données

In [4]:
# Vérification des valeurs nulles
print("🔍 Valeurs nulles par colonne :")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

🔍 Valeurs nulles par colonne :
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|        0|        0|       1454|       0|          0|        0|    135080|      0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+



In [5]:
# Nettoyage :
# - Suppression des lignes sans CustomerID
# - Suppression des quantités négatives (retours / annulations)
# - Suppression des prix nuls ou négatifs
# - Suppression des doublons

df_clean = df \
    .filter(F.col("CustomerID").isNotNull()) \
    .filter(F.col("Quantity") > 0) \
    .filter(F.col("UnitPrice") > 0) \
    .dropDuplicates()

# CustomerID en entier (le CSV le stocke en flottant : 17850.0 -> 17850)
df_clean = df_clean.withColumn("CustomerID", F.col("CustomerID").cast("int"))

# Ajout d'une colonne TotalPrice (montant de la ligne)
df_clean = df_clean.withColumn("TotalPrice", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))

# Conversion de la date (format du CSV : yyyy-MM-dd HH:mm:ss)
df_clean = df_clean.withColumn("InvoiceDate", F.to_timestamp("InvoiceDate", "yyyy-MM-dd HH:mm:ss"))
df_clean = df_clean.withColumn("Month", F.month("InvoiceDate"))
df_clean = df_clean.withColumn("Year", F.year("InvoiceDate"))

print(f"✅ Après nettoyage : {df_clean.count()} lignes")
df_clean.show(5, truncate=False)

✅ Après nettoyage : 392692 lignes
+---------+---------+----------------------------------+--------+-------------------+---------+----------+--------------+----------+-----+----+
|InvoiceNo|StockCode|Description                       |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |TotalPrice|Month|Year|
+---------+---------+----------------------------------+--------+-------------------+---------+----------+--------------+----------+-----+----+
|536408   |84879    |ASSORTED COLOUR BIRD ORNAMENT     |8       |2010-12-01 11:41:00|1.69     |14307     |United Kingdom|13.52     |12   |2010|
|536409   |22074    |6 RIBBONS SHIMMERING PINKS        |1       |2010-12-01 11:45:00|1.65     |17908     |United Kingdom|1.65      |12   |2010|
|536409   |90199C   |5 STRAND GLASS NECKLACE CRYSTAL   |2       |2010-12-01 11:45:00|6.35     |17908     |United Kingdom|12.7      |12   |2010|
|536488   |22150    |3 STRIPEY MICE FELTCRAFT          |1       |2010-12-01 12:31:00|1.95     |17897  

## 4️⃣ Analyses des ventes

In [6]:
# ── Chiffre d'affaires total ──
ca_total = df_clean.agg(F.round(F.sum("TotalPrice"), 2).alias("CA_Total")).collect()[0][0]
print(f"💰 Chiffre d'affaires total : {ca_total} £")

💰 Chiffre d'affaires total : 8887208.89 £


In [7]:
# ── Top 10 produits les plus vendus (en quantité) ──
print("🏆 Top 10 produits les plus vendus :")
df_clean.groupBy("StockCode", "Description") \
    .agg(F.sum("Quantity").alias("Total_Vendu")) \
    .orderBy(F.desc("Total_Vendu")) \
    .show(10, truncate=False)

🏆 Top 10 produits les plus vendus :
+---------+----------------------------------+-----------+
|StockCode|Description                       |Total_Vendu|
+---------+----------------------------------+-----------+
|23843    |PAPER CRAFT , LITTLE BIRDIE       |80995      |
|23166    |MEDIUM CERAMIC TOP STORAGE JAR    |77916      |
|84077    |WORLD WAR 2 GLIDERS ASSTD DESIGNS |54319      |
|85099B   |JUMBO BAG RED RETROSPOT           |46078      |
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER|36706      |
|84879    |ASSORTED COLOUR BIRD ORNAMENT     |35263      |
|21212    |PACK OF 72 RETROSPOT CAKE CASES   |33670      |
|22197    |POPCORN HOLDER                    |30919      |
|23084    |RABBIT NIGHT LIGHT                |27153      |
|22492    |MINI PAINT SET VINTAGE            |26076      |
+---------+----------------------------------+-----------+
only showing top 10 rows



In [8]:
# ── Top 10 produits par chiffre d'affaires ──
print("💵 Top 10 produits par CA :")
df_clean.groupBy("StockCode", "Description") \
    .agg(F.round(F.sum("TotalPrice"), 2).alias("CA")) \
    .orderBy(F.desc("CA")) \
    .show(10, truncate=False)

💵 Top 10 produits par CA :
+---------+----------------------------------+---------+
|StockCode|Description                       |CA       |
+---------+----------------------------------+---------+
|23843    |PAPER CRAFT , LITTLE BIRDIE       |168469.6 |
|22423    |REGENCY CAKESTAND 3 TIER          |142264.75|
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER|100392.1 |
|85099B   |JUMBO BAG RED RETROSPOT           |85040.54 |
|23166    |MEDIUM CERAMIC TOP STORAGE JAR    |81416.73 |
|POST     |POSTAGE                           |77803.96 |
|47566    |PARTY BUNTING                     |68785.23 |
|84879    |ASSORTED COLOUR BIRD ORNAMENT     |56413.03 |
|M        |Manual                            |53419.93 |
|23084    |RABBIT NIGHT LIGHT                |51251.24 |
+---------+----------------------------------+---------+
only showing top 10 rows



In [9]:
# ── Chiffre d'affaires par pays ──
print("🌍 CA par pays (Top 10) :")
df_clean.groupBy("Country") \
    .agg(F.round(F.sum("TotalPrice"), 2).alias("CA")) \
    .orderBy(F.desc("CA")) \
    .show(10, truncate=False)

🌍 CA par pays (Top 10) :
+--------------+----------+
|Country       |CA        |
+--------------+----------+
|United Kingdom|7285024.64|
|Netherlands   |285446.34 |
|EIRE          |265262.46 |
|Germany       |228678.4  |
|France        |208934.31 |
|Australia     |138453.81 |
|Spain         |61558.56  |
|Switzerland   |56443.95  |
|Belgium       |41196.34  |
|Sweden        |38367.83  |
+--------------+----------+
only showing top 10 rows



In [10]:
# ── Évolution mensuelle du CA ──
print("📅 Évolution mensuelle du CA :")
df_clean.groupBy("Year", "Month") \
    .agg(F.round(F.sum("TotalPrice"), 2).alias("CA_Mensuel")) \
    .orderBy("Year", "Month") \
    .show(20)

📅 Évolution mensuelle du CA :
+----+-----+----------+
|Year|Month|CA_Mensuel|
+----+-----+----------+
|2010|   12| 570422.73|
|2011|    1| 568101.31|
|2011|    2| 446084.92|
|2011|    3| 594081.76|
|2011|    4| 468374.33|
|2011|    5| 677355.15|
|2011|    6| 660046.05|
|2011|    7|  598962.9|
|2011|    8| 644051.04|
|2011|    9|  950690.2|
|2011|   10|1035642.45|
|2011|   11|1156205.61|
|2011|   12| 517190.44|
+----+-----+----------+



In [11]:
# ── Top 10 clients par CA ──
print("👤 Top 10 clients par CA :")
df_clean.groupBy("CustomerID") \
    .agg(F.round(F.sum("TotalPrice"), 2).alias("CA_Client")) \
    .orderBy(F.desc("CA_Client")) \
    .show(10)

👤 Top 10 clients par CA :
+----------+---------+
|CustomerID|CA_Client|
+----------+---------+
|     14646|280206.02|
|     18102| 259657.3|
|     17450|194390.79|
|     16446| 168472.5|
|     14911|143711.17|
|     12415|124914.53|
|     14156|117210.08|
|     17511| 91062.38|
|     16029| 80850.84|
|     12346|  77183.6|
+----------+---------+
only showing top 10 rows



## 5️⃣ Indicateurs clés (KPIs)

In [12]:
# ── Nombre de commandes uniques ──
nb_commandes = df_clean.select("InvoiceNo").distinct().count()
print(f"🧾 Nombre de commandes uniques : {nb_commandes}")

# ── Nombre de clients uniques ──
nb_clients = df_clean.select("CustomerID").distinct().count()
print(f"👥 Nombre de clients uniques : {nb_clients}")

# ── Nombre de produits uniques ──
nb_produits = df_clean.select("StockCode").distinct().count()
print(f"📦 Nombre de produits uniques : {nb_produits}")

# ── Panier moyen par commande ──
panier_moyen = df_clean.groupBy("InvoiceNo") \
    .agg(F.sum("TotalPrice").alias("Total_Commande")) \
    .agg(F.round(F.avg("Total_Commande"), 2).alias("Panier_Moyen")) \
    .collect()[0][0]
print(f"🛒 Panier moyen : {panier_moyen} £")

# ── Nombre moyen d'articles par commande ──
articles_moyen = df_clean.groupBy("InvoiceNo") \
    .agg(F.sum("Quantity").alias("NbArticles")) \
    .agg(F.round(F.avg("NbArticles"), 1).alias("Moy_Articles")) \
    .collect()[0][0]
print(f"📦 Nombre moyen d'articles par commande : {articles_moyen}")

🧾 Nombre de commandes uniques : 18532
👥 Nombre de clients uniques : 4338
📦 Nombre de produits uniques : 3665
🛒 Panier moyen : 479.56 £
📦 Nombre moyen d'articles par commande : 278.0


## 6️⃣ Stockage sur HDFS

In [13]:
# ── (Partie 2) Stockage du fichier BRUT du dataset sur HDFS ──
# On dépose le dataset complet (avant nettoyage) sur le cluster HDFS,
# comme demandé dans l'énoncé. On le sauvegarde en CSV partitionné.
hdfs_raw_path = "hdfs://namenode:9000/data/raw/online_retail"

df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(hdfs_raw_path)

print(f"✅ Dataset brut stocké sur HDFS : {hdfs_raw_path}")

✅ Dataset brut stocké sur HDFS : hdfs://namenode:9000/data/raw/online_retail


In [14]:
# Sauvegarde des données nettoyées sur HDFS
hdfs_path = "hdfs://namenode:9000/data/online_retail_clean"

df_clean.write \
    .mode("overwrite") \
    .parquet(hdfs_path)

print(f"✅ Données sauvegardées sur HDFS : {hdfs_path}")

✅ Données sauvegardées sur HDFS : hdfs://namenode:9000/data/online_retail_clean


In [15]:
# Vérification : relecture depuis HDFS
df_hdfs = spark.read.parquet(hdfs_path)
print(f"✅ Lecture depuis HDFS : {df_hdfs.count()} lignes")
df_hdfs.show(5)

✅ Lecture depuis HDFS : 392692 lignes
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+----------+-----+----+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|TotalPrice|Month|Year|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+----------+-----+----+
|   536375|    82482|WOODEN PICTURE FR...|       6|2010-12-01 09:32:00|      2.1|     17850|United Kingdom|      12.6|   12|2010|
|   536378|    20725|LUNCH BAG RED RET...|      10|2010-12-01 09:37:00|     1.65|     14688|United Kingdom|      16.5|   12|2010|
|   536390|    22654|  DELUXE SEWING KIT |      40|2010-12-01 10:19:00|     4.95|     17511|United Kingdom|     198.0|   12|2010|
|   536464|   90059C|DIAMANTE HAIR GRI...|       1|2010-12-01 12:23:00|     1.65|     17968|United Kingdom|      1.65|   12|2010|
|   536488|    20878|SET/9 CHRISTMAS T...|       2|2

## 7️⃣ Fin de session

In [16]:
spark.stop()
print("✅ Session Spark arrêtée.")

✅ Session Spark arrêtée.
